# Lecture 5: Classification Fundamentals — Answer Key
### NLP Course 2027 — Week 03

---
Complete answers for all four exercises in Lecture 5.

In [ ]:
import nltk, random, numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
nltk.download('names', quiet=True)
nltk.download('movie_reviews', quiet=True)
print('Ready')

---
## Exercise 5.1 — Naive Bayes Gender Classifier

**Task:** Train a Naive Bayes classifier on NLTK `names` corpus. Use last 2 chars as feature plus additional features.

**Key concept:** Feature engineering drives Naive Bayes performance far more than model choice. Adding first letter, last letter, suffix patterns, and name length all help.

In [ ]:
from nltk.corpus import names

def gender_features(name):
    """Rich feature set for gender classification."""
    name = name.lower()
    return {
        'last_letter':  name[-1],
        'last_two':     name[-2:],
        'last_three':   name[-3:] if len(name) >= 3 else name,
        'first_letter': name[0],
        'first_two':    name[:2],
        'length':       len(name),
        'ends_in_vowel': name[-1] in 'aeiou',
        'ends_in_a':    name[-1] == 'a',
        'ends_in_ie':   name[-2:] == 'ie',
        'ends_in_en':   name[-2:] == 'en',
    }

labeled = ([(n, 'male')   for n in names.words('male.txt')] +
           [(n, 'female') for n in names.words('female.txt')])
random.seed(42)
random.shuffle(labeled)

featuresets = [(gender_features(n), g) for n, g in labeled]
train, test = featuresets[500:], featuresets[:500]

clf = nltk.NaiveBayesClassifier.train(train)
print(f'Accuracy: {nltk.classify.accuracy(clf, test):.4f}')
print()
clf.show_most_informative_features(10)

**Expected accuracy:** ~80–83%. The most informative features are typically `last_two=na` (female) and `last_two=on` (male). Adding `last_three` and suffix patterns pushes accuracy above the 2-feature baseline (~78%).

---
## Exercise 5.2 — CountVectorizer vs TfidfVectorizer

**Task:** Compare on 4-category 20 Newsgroups. Does TF-IDF help?

**Key concept:** TF-IDF downweights common words (like 'the', 'is') that appear in all documents. For multi-class classification, this typically improves accuracy by 2–5%.

In [ ]:
categories = ['rec.sport.hockey', 'sci.space', 'comp.graphics', 'talk.politics.guns']
train_data = fetch_20newsgroups(subset='train', categories=categories,
                                remove=('headers', 'footers', 'quotes'))
test_data  = fetch_20newsgroups(subset='test',  categories=categories,
                                remove=('headers', 'footers', 'quotes'))

for name, vec in [('CountVectorizer', CountVectorizer(min_df=2)),
                  ('TfidfVectorizer', TfidfVectorizer(min_df=2))]:
    pipe = Pipeline([('vec', vec), ('clf', MultinomialNB())])
    pipe.fit(train_data.data, train_data.target)
    acc = accuracy_score(test_data.target, pipe.predict(test_data.data))
    print(f'{name}: {acc:.4f}')

print()
print('TF-IDF usually wins by ~3–5% because it suppresses high-frequency non-informative words.')

---
## Exercise 5.3 — 5-Fold Cross-Validation

**Task:** Use `cross_val_score(cv=5)`. Report mean ± std.

**Key concept:** Cross-validation gives a more reliable accuracy estimate than a single train/test split. The std tells you how stable the model is across different data splits.

In [ ]:
# Use the full training set for cross-validation
all_data = fetch_20newsgroups(subset='train', categories=categories,
                              remove=('headers', 'footers', 'quotes'))

pipe = Pipeline([('tfidf', TfidfVectorizer(min_df=2)), ('clf', MultinomialNB())])
scores = cross_val_score(pipe, all_data.data, all_data.target, cv=5, scoring='accuracy')

print(f'5-Fold CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}')
print(f'Per-fold scores:    {[f"{s:.4f}" for s in scores]}')

---
## Exercise 5.4 — Unigrams vs Unigrams + Bigrams

**Task:** Compare `ngram_range=(1,1)` vs `(1,2)`. Does accuracy improve?

**Key concept:** Bigrams capture local context ('not good' vs 'good'). They increase vocabulary size substantially but often improve accuracy on topic classification tasks.

In [ ]:
for label, ngram_range in [('Unigrams only (1,1)', (1, 1)),
                            ('Unigrams + Bigrams (1,2)', (1, 2))]:
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(min_df=2, ngram_range=ngram_range)),
        ('clf', LogisticRegression(max_iter=1000)),
    ])
    pipe.fit(train_data.data, train_data.target)
    acc = accuracy_score(test_data.target, pipe.predict(test_data.data))
    vocab_size = len(pipe.named_steps['tfidf'].vocabulary_)
    print(f'{label}: accuracy={acc:.4f}, vocab_size={vocab_size:,}')

print()
print('Bigrams improve accuracy but at the cost of a much larger feature space.')
print('Consider min_df=3 or max_features to control vocabulary size.')

---
## Summary

| Exercise | Key Concept |
|----------|-------------|
| 5.1 | Feature engineering > model choice for Naive Bayes; suffix patterns are highly predictive |
| 5.2 | TF-IDF outperforms raw counts by downweighting common non-informative words |
| 5.3 | Cross-validation gives stable accuracy estimates; low std = robust model |
| 5.4 | Bigrams improve accuracy but inflate vocabulary; use `min_df` to prune |

---
*NLP Course 2027 — Week 03*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**